# Training without the video holds

**Decision (Hassan and Abdullah, 20 Nov 2026):** the 1,345 holds pseudo-labelled from the PSL-CFRT videos
(signers 1–5) are dropped from training: many frames are blurred or distorted and their labels are inferred,
not checked. Training keeps:

* **calibration takes** (291, 30 static letters, Hassan and Abdullah),
* **Kamran's letter images** (2,902 after clean-up, 38 letters; private),
* **official PSL images** (8 screenshots from the PSL website: ا ب ل ک; private).

This notebook measures what that does. **Old** = video holds + the three sources above;
**new** = the three sources only. Same models and settings as before: the shipped SVM (scored by
its own `predict`, as in the browser), ExtraTrees, and the 11 Nov ensemble (SVM + MLP + ExtraTrees + LDA).

**Tests.** Nothing is tuned, so no validation set is needed.
1. **New people:** the five video signers, *as test data only*. They are still the only recordings of
   people the models never saw. Their labels are the noisy pseudo-labels, so these numbers measure
   agreement with them; read the old/new difference, not the level. Old is leave-one-signer-out (the test
   signer's holds never in training); new never trains on any video hold.
2. **Calibrated user:** held-out calibration takes (60/20/20 per letter, `splits.calibration_split`,
   five seeds; the validation takes join training).
3. **Kamran's validation split:** known to be inflated (his val images are near copies of his train
   images, `kamran_image_dataset.ipynb`).
4. **Official images:** the only test labels known to be right, but 8 images. Left out of training here.

New training has no example of آ ؤ ئ (neither calibration nor Kamran's set has them), so it cannot
recognise them. Static-letter scores are unaffected by this.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import os
import numpy as np, pandas as pd
from scipy.stats import binomtest
from sklearn.metrics import recall_score

import evaluation as ev, models, splits, image_dataset
from hand_geometry import hand_features

OUT = "results/2026-11-20_no_video_training"
os.makedirs(OUT, exist_ok=True)
DYN = set(splits.DYNAMIC_LETTERS)

P, M = ev.load_dataset(include_users=True)
user = splits.is_user(M.signer)
F, y, g = hand_features(P), M.letter.to_numpy(), M.signer.to_numpy()

PK, MK = image_dataset.load()
keep = ((MK.detected == 1) & ~MK.duplicated("sha1")).to_numpy()
FK, yK, sK = hand_features(PK[keep]), MK.letter.to_numpy()[keep], MK.split.to_numpy()[keep]

PO, MO = image_dataset.official()
FO, yO = hand_features(PO), MO.letter.to_numpy()

V = ~user
print(f"video holds {V.sum()} ({len(set(y[V]))} letters) | calibration {user.sum()} ({len(set(y[user]))} letters)"
      f" | Kamran {len(yK)} ({len(set(yK))}) | official {len(yO)} ({sorted(set(yO))})")
print("letters only the video holds have:", sorted(set(y[V]) - set(y[user]) - set(yK)))

video holds 1345 (40 letters) | calibration 291 (30 letters) | Kamran 2902 (38) | official 8 (['ا', 'ب', 'ل', 'ک'])
letters only the video holds have: ['آ', 'ؤ', 'ئ']


In [2]:
MODELS = {
    "SVM (shipped)": (lambda: ev.svm(), "predict"),
    "ExtraTrees": (lambda: models.extra_trees(0), "proba"),
    "ensemble SVM+MLP+ET+LDA": (lambda: models.ensemble(0), "proba"),
}

def fit_predict(name, Ftr, ytr, Fte):
    make, how = MODELS[name]
    m = make().fit(Ftr, ytr)
    return m.predict(Fte) if how == "predict" else m.classes_[m.predict_proba(Fte).argmax(1)]

def static_mask(yy, static):
    return ~np.isin(yy, list(DYN)) if static else np.ones(len(yy), bool)

def stack(*parts):
    parts = [p for p in parts if p is not None and len(p[1])]
    return np.vstack([p[0] for p in parts]), np.concatenate([p[1] for p in parts])

def macro(yt, yp):
    return 100 * recall_score(yt, yp, average="macro", labels=sorted(set(yt)))

## 1. New people (the five video signers, test only)

In [3]:
rows = []
for static in (False, True):
    tag = "30 static letters" if static else "all letters"
    s_tr = lambda yy: static_mask(yy, static)
    cal = (F[user][s_tr(y[user])], y[user][s_tr(y[user])])
    kam = (FK[s_tr(yK)], yK[s_tr(yK)])
    off = (FO[s_tr(yO)], yO[s_tr(yO)])
    for name in MODELS:
        Fn, yn = stack(cal, kam, off)
        mn = MODELS[name][0]().fit(Fn, yn)                 # new: one model, never sees a video hold
        for s in sorted(set(g[V])):
            te = (g == s) & static_mask(y, static)
            other = V & (g != s) & static_mask(y, static)
            Fo, yo = stack((F[other], y[other]), cal, kam, off)
            p_old = fit_predict(name, Fo, yo, F[te])
            p_new = mn.predict(F[te]) if MODELS[name][1] == "predict" else mn.classes_[mn.predict_proba(F[te]).argmax(1)]
            for arm, p in (("old (with video)", p_old), ("new (no video)", p_new)):
                rows.append({"letters": tag, "model": name, "test signer": s, "training": arm,
                             "accuracy %": 100 * (p == y[te]).mean(), "macro recall %": macro(y[te], p)})
N = pd.DataFrame(rows)
t1 = N.pivot_table(index=["letters", "model"], columns="training", values=["accuracy %", "macro recall %"]).round(1)
d = N.pivot_table(index=["letters", "model", "test signer"], columns="training", values="accuracy %")
d["change"] = d["new (no video)"] - d["old (with video)"]
t1[("change", "pts")] = d.groupby(level=[0, 1])["change"].mean().round(1)
t1[("change", "signers better")] = d.groupby(level=[0, 1])["change"].apply(lambda c: f"{(c > 0).sum()} of {len(c)}")
t1

accuracy %                   \
training                                  new (no video) old (with video)   
letters           model                                                     
30 static letters ExtraTrees                        39.6             42.2   
                  SVM (shipped)                     38.0             42.9   
                  ensemble SVM+MLP+ET+LDA           40.0             43.6   
all letters       ExtraTrees                        31.1             34.8   
                  SVM (shipped)                     28.8             33.7   
                  ensemble SVM+MLP+ET+LDA           30.2             35.4   

                                          macro recall %                   \
training                                  new (no video) old (with video)   
letters           model                                                     
30 static letters ExtraTrees                        39.7             42.6   
                  SVM (shipped)                     39.1             43.5   
                  ensemble SVM+MLP+ET+LDA           41.3             42.5   
all letters       ExtraTrees                        32.9             34.5   
                  SVM (shipped)                     31.8             34.9   
                  ensemble SVM+MLP+ET+LDA           32.0             34.4   

                                          change                 
training                                     pts signers better  
letters           model                                          
30 static letters ExtraTrees                -2.6         0 of 5  
                  SVM (shipped)             -4.9         0 of 5  
                  ensemble SVM+MLP+ET+LDA   -3.7         0 of 5  
all letters       ExtraTrees                -3.7         0 of 5  
                  SVM (shipped)             -4.9         1 of 5  
                  ensemble SVM+MLP+ET+LDA   -5.2         0 of 5

In [4]:
d.round(1)

training                                               new (no video)  \
letters           model                   test signer                   
30 static letters ExtraTrees              signer1-F              34.2   
                                          signer2-F              40.6   
                                          signer3-F              36.9   
                                          signer4-L              42.9   
                                          signer5-R              43.5   
                  SVM (shipped)           signer1-F              33.3   
                                          signer2-F              38.4   
                                          signer3-F              35.5   
                                          signer4-L              42.4   
                                          signer5-R              40.3   
                  ensemble SVM+MLP+ET+LDA signer1-F              32.5   
                                          signer2-F              42.8   
                                          signer3-F              35.5   
                                          signer4-L              44.6   
                                          signer5-R              44.4   
all letters       ExtraTrees              signer1-F              28.6   
                                          signer2-F              31.1   
                                          signer3-F              28.3   
                                          signer4-L              35.9   
                                          signer5-R              31.6   
                  SVM (shipped)           signer1-F              26.7   
                                          signer2-F              30.5   
                                          signer3-F              24.0   
                                          signer4-L              33.6   
                                          signer5-R              29.1   
                  ensemble SVM+MLP+ET+LDA signer1-F              27.3   
                                          signer2-F              32.8   
                                          signer3-F              26.9   
                                          signer4-L              34.2   
                                          signer5-R              29.7   

training                                               old (with video)  \
letters           model                   test signer                     
30 static letters ExtraTrees              signer1-F                35.9   
                                          signer2-F                41.9   
                                          signer3-F                41.5   
                                          signer4-L                44.2   
                                          signer5-R                47.6   
                  SVM (shipped)           signer1-F                38.5   
                                          signer2-F                43.2   
                                          signer3-F                40.1   
                                          signer4-L                45.1   
                                          signer5-R                47.6   
                  ensemble SVM+MLP+ET+LDA signer1-F                36.8   
                                          signer2-F                45.0   
                                          signer3-F                41.9   
                                          signer4-L                46.0   
                                          signer5-R                48.4   
all letters       ExtraTrees              signer1-F                29.9   
                                          signer2-F                37.1   
                                          signer3-F                33.7   
                                          signer4-L                37.3   
                                          signer5-R                36.1   
             

## 2. Calibrated user (held-out calibration takes, five splits)

In [5]:
rows = []
Mu = M[user].reset_index(drop=True)
Fu, yu = F[user], y[user]
for seed in range(5):
    role = splits.calibration_split(Mu, seed)
    tr, te = role != "test", role == "test"
    for name in MODELS:
        for arm, video in (("old (with video)", True), ("new (no video)", False)):
            Ftr, ytr = stack((Fu[tr], yu[tr]), (FK, yK), (FO, yO), (F[V], y[V]) if video else None)
            p = fit_predict(name, Ftr, ytr, Fu[te])
            rows.append({"model": name, "training": arm, "seed": seed, "correct": int((p == yu[te]).sum()),
                         "n": int(te.sum()), "macro recall %": macro(yu[te], p)})
Cc = pd.DataFrame(rows)
t2 = Cc.groupby(["model", "training"]).agg(correct=("correct", "sum"), n=("n", "sum"),
                                            macro=("macro recall %", "mean"))
t2["accuracy %"] = (100 * t2.correct / t2.n).round(1)
t2["macro recall %"] = t2.macro.round(1)
t2[["correct", "n", "accuracy %", "macro recall %"]]

correct    n  accuracy %  \
model                   training                                     
ExtraTrees              new (no video)        262  290        90.3   
                        old (with video)      263  290        90.7   
SVM (shipped)           new (no video)        271  290        93.4   
                        old (with video)      267  290        92.1   
ensemble SVM+MLP+ET+LDA new (no video)        258  290        89.0   
                        old (with video)      255  290        87.9   

                                          macro recall %  
model                   training                          
ExtraTrees              new (no video)              90.7  
                        old (with video)            91.0  
SVM (shipped)           new (no video)              93.7  
                        old (with video)            92.3  
ensemble SVM+MLP+ET+LDA new (no video)              89.3  
                        old (with video)            88.3

## 3. Kamran's validation split (inflated: near copies of his training images)

In [6]:
rows = []
tr, va = sK == "train", sK == "val"
for name in MODELS:
    for arm, video in (("old (with video)", True), ("new (no video)", False)):
        Ftr, ytr = stack((FK[tr], yK[tr]), (F[user], y[user]), (FO, yO), (F[V], y[V]) if video else None)
        p = fit_predict(name, Ftr, ytr, FK[va])
        rows.append({"model": name, "training": arm, "accuracy %": round(100 * (p == yK[va]).mean(), 1),
                     "macro recall %": round(macro(yK[va], p), 1), "n": int(va.sum())})
t3 = pd.DataFrame(rows)
t3

,model,training,accuracy %,macro recall %,n
0,SVM (shipped),old (with video),81.2,81.1,894
1,SVM (shipped),new (no video),82.4,82.4,894
2,ExtraTrees,old (with video),93.2,93.0,894
3,ExtraTrees,new (no video),93.7,93.7,894
4,ensemble SVM+MLP+ET+LDA,old (with video),89.1,88.9,894
5,ensemble SVM+MLP+ET+LDA,new (no video),88.4,87.9,894


## 4. Official PSL images (8, left out of training)

In [7]:
rows = []
for name in MODELS:
    for arm, video in (("old (with video)", True), ("new (no video)", False)):
        Ftr, ytr = stack((F[user], y[user]), (FK, yK), (F[V], y[V]) if video else None)
        p = fit_predict(name, Ftr, ytr, FO)
        rows.append({"model": name, "training": arm, "correct of 8": int((p == yO).sum()),
                     "predictions": " ".join(f"{a}->{b}" for a, b in zip(yO, p))})
t4 = pd.DataFrame(rows)
t4

,model,training,correct of 8,predictions
0,SVM (shipped),old (with video),6,ا->ا ا->ا ب->ب ب->ب ل->ل ل->ل ک->ڑ ک->ڑ
1,SVM (shipped),new (no video),6,ا->ا ا->ا ب->ب ب->ب ل->ل ل->ل ک->ڑ ک->ڑ
2,ExtraTrees,old (with video),6,ا->ا ا->ا ب->ب ب->ب ل->ل ل->ل ک->ڑ ک->ڑ
3,ExtraTrees,new (no video),6,ا->ا ا->ا ب->ب ب->ب ل->ل ل->ل ک->ڑ ک->ڑ
4,ensemble SVM+MLP+ET+LDA,old (with video),8,ا->ا ا->ا ب->ب ب->ب ل->ل ل->ل ک->ک ک->ک
5,ensemble SVM+MLP+ET+LDA,new (no video),6,ا->ا ا->ا ب->ب ب->ب ل->ل ل->ل ک->ڑ ک->ڑ


In [8]:
N.to_csv(f"{OUT}/new_people_per_signer.csv", index=False, encoding="utf-8")
Cc.to_csv(f"{OUT}/calibrated_user_per_split.csv", index=False, encoding="utf-8")
t3.to_csv(f"{OUT}/kamran_val.csv", index=False, encoding="utf-8")
t4.drop(columns="predictions").to_csv(f"{OUT}/official.csv", index=False, encoding="utf-8")
print("written to", OUT)

written to results/2026-11-20_no_video_training
